
# Лабораторная работа №1
## Предобработка набора данных UNSW-NB15 для задачи обнаружения сетевых вторжений

**Дисциплина:** Модели и методы машинного обучения  
**Преподаватель:** Вовик А.Г.  
**Выполнила:** студентка гр. МФБ2631 Дарбека Е.

### Цель работы

Освоить основные этапы предварительной обработки данных средствами **Pandas** и **Scikit-learn** на реальном наборе данных из области информационной безопасности и подготовить его к последующему обучению моделей классификации.

В работе выполняются:

1. чтение и первичная проверка данных;
2. поиск и обработка пропущенных значений;
3. изменение структуры и работа со строками с помощью регулярных выражений;
4. кодирование категориальных признаков;
5. визуализация данных;
6. создание и отбор признаков;
7. проверка сбалансированности классов;
8. разделение на `X`, `y`, `train`, `test`;
9. нормализация признаков.

> В этом ноутбуке используется официальная обучающая выборка **UNSW-NB15**.  
> При каждом запуске набора ячеек CSV скачивается заново во временный каталог.  
> Временные файлы удаляются при завершении ядра Jupyter или вручную последней ячейкой.



## 0. Подготовка окружения

Если библиотеки ещё не установлены, выполните в терминале VS Code:

```bash
pip install requests pandas numpy matplotlib seaborn scikit-learn
```

В самом Jupyter можно также использовать:

```python
%pip install requests pandas numpy matplotlib seaborn scikit-learn
```

После установки желательно перезапустить ядро.


In [ ]:

import os
import io
import re
import atexit
import shutil
import tempfile
from pathlib import Path

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFE
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)



## Временный рабочий каталог

Набор данных и все подготовленные CSV создаются во **временной папке**.

Это позволяет не оставлять большие файлы после завершения лабораторной.  
При остановке ядра Jupyter функция `cleanup_temp_dir()` автоматически удалит эту папку.


In [ ]:

WORK_DIR = Path(tempfile.mkdtemp(prefix="lab1_unsw_"))
SOURCE_FILENAME = "UNSW_NB15_training-set.csv"
SOURCE_CSV = WORK_DIR / SOURCE_FILENAME

DATASET_URLS = [
    "https://huggingface.co/datasets/Mouwiya/UNSW-NB15/resolve/main/UNSW_NB15_training-set.csv",
    "https://raw.githubusercontent.com/Nir-J/ML-Projects/master/UNSW-Network_Packet_Classification/UNSW_NB15_training-set.csv",
]

def cleanup_temp_dir():
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR, ignore_errors=True)
        print(f"Временный каталог удалён: {WORK_DIR}")

atexit.register(cleanup_temp_dir)

print("Временный каталог:", WORK_DIR)



## Загрузка UNSW-NB15

Используются два источника:

1. Hugging Face — основной;
2. GitHub — резервный.

Если первый источник недоступен, программа автоматически переходит ко второму.

Дополнительно проверяется размер скачанного файла: слишком маленький файл может означать, что вместо CSV сервер вернул HTML-страницу или сообщение об ошибке.


In [ ]:

def download_dataset():
    min_size = 5_000_000
    last_error = None

    for url in DATASET_URLS:
        print(f"\nИсточник: {url}")
        temp_file = SOURCE_CSV.with_suffix(".part")

        try:
            with requests.get(
                url,
                stream=True,
                timeout=(15, 180),
                headers={"User-Agent": "Lab1-UNSW-NB15/1.0"},
            ) as response:
                print("HTTP status:", response.status_code)
                response.raise_for_status()

                total = int(response.headers.get("content-length", 0))
                downloaded = 0

                with open(temp_file, "wb") as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if not chunk:
                            continue

                        file.write(chunk)
                        downloaded += len(chunk)

                        if total:
                            print(
                                f"\rСкачано: {downloaded / 1024**2:.1f} / "
                                f"{total / 1024**2:.1f} MB",
                                end=""
                            )

            print()

            file_size = temp_file.stat().st_size
            print(f"Размер файла: {file_size / 1024**2:.2f} MB")

            if file_size < min_size:
                raise ValueError(
                    f"Файл подозрительно мал: {file_size:,} байт."
                )

            os.replace(temp_file, SOURCE_CSV)
            print("CSV создан:", SOURCE_CSV)
            return SOURCE_CSV

        except Exception as error:
            last_error = error
            print("Ошибка источника:", error)
            temp_file.unlink(missing_ok=True)

    raise RuntimeError(
        "Не удалось скачать UNSW-NB15 ни из одного источника. "
        f"Последняя ошибка: {last_error}"
    )

download_dataset()



# 1. Pandas: чтение файла и проверка корректности

Файл читается методом `pd.read_csv()` в объект `DataFrame`.

После загрузки проверяем:

- первые строки через `head()`;
- размер через `shape`;
- названия столбцов;
- типы данных;
- `info()`;
- статистики `describe()`;
- количество дубликатов;
- уникальность `id`.

Целевой признак:

- `label = 0` — нормальный трафик;
- `label = 1` — атака.

Дополнительный признак `attack_cat` содержит категорию атаки.


In [ ]:

df = pd.read_csv(SOURCE_CSV)

required_columns = {
    "id", "proto", "service", "state", "attack_cat", "label",
    "dur", "sbytes", "dbytes", "spkts", "dpkts",
    "sttl", "dttl", "sload", "dload"
}

missing_columns = sorted(required_columns - set(df.columns))

if missing_columns:
    raise ValueError(
        "В CSV отсутствуют ожидаемые столбцы: "
        + ", ".join(missing_columns)
    )

print("Размер набора данных:", df.shape)
display(df.head())


In [ ]:

print("Названия столбцов:")
print(df.columns.tolist())

print("\nКоличество столбцов каждого типа:")
display(df.dtypes.value_counts().to_frame("count"))

print("\nИнформация DataFrame:")
df.info()


In [ ]:

print("Статистическое описание числовых признаков:")
display(df.describe().T)

print("Дубликатов строк:", df.duplicated().sum())
print("id уникален:", df["id"].is_unique)

cat_cols = df.select_dtypes(include="object").columns.tolist()
print("Категориальные столбцы:", cat_cols)


In [ ]:

print("Распределение attack_cat:")
display(df["attack_cat"].value_counts().to_frame("count"))

print("Распределение label:")
display(df["label"].value_counts().to_frame("count"))



# 2. Проверка и обработка пустых значений

Сначала ищем обычные `NaN`.

Однако отсутствие явных `NaN` ещё не гарантирует отсутствие пропусков.  
В `service` значение `"-"` означает, что прикладной сервис определить не удалось. Это **скрытый пропуск**.

Удалять такие строки нецелесообразно, поскольку их очень много.  
Заполнять наиболее частым сервисом тоже некорректно: неизвестный сервис имеет собственный смысл.

Поэтому используется отдельная категория:

```text
unknown
```


In [ ]:

print("Явных NaN во всём DataFrame:", df.isna().sum().sum())

for col in cat_cols:
    print(
        f"{col}: {df[col].nunique(dropna=False)} уникальных значений"
    )
    display(df[col].value_counts(dropna=False).head(10).to_frame("count"))


In [ ]:

n_dash = (df["service"] == "-").sum()

print(
    f"Скрытых пропусков '-' в service: "
    f"{n_dash:,} ({n_dash / len(df):.2%})"
)

df["service"] = df["service"].replace("-", np.nan)

print(
    "Доля атак среди неизвестных сервисов:",
    round(df.loc[df["service"].isna(), "label"].mean(), 3)
)

print(
    "Доля атак среди известных сервисов:",
    round(df.loc[df["service"].notna(), "label"].mean(), 3)
)



### Визуализация пропусков

Для самой тепловой карты достаточно первых 5000 строк: статистика пропусков всё равно считается по полному набору, а отображение всех 175 тысяч строк не добавляет полезной информации и сильно нагружает графическую подсистему.


In [ ]:

missing_sample = df.head(min(5000, len(df)))

plt.figure(figsize=(12, 5))
sns.heatmap(
    missing_sample.isna(),
    cbar=False,
    yticklabels=False,
    cmap="viridis"
)
plt.title("Рис. 1. Тепловая карта пропусков")
plt.xlabel("Признаки")
plt.show()


In [ ]:

df["service"] = df["service"].fillna("unknown")

print("NaN после обработки:", df.isna().sum().sum())
print("Количество 'unknown':", (df["service"] == "unknown").sum())



# 3. Изменение структуры и регулярные выражения

Для демонстрации работы с текстом создаётся составной признак:

```text
proto|service|state
```

Например:

```text
tcp|http|FIN
```

Затем он разбирается обратно регулярным выражением.

Разделитель `|` выбран специально: он не должен конфликтовать со значениями самих признаков.


In [ ]:

df = df.drop(columns=["id"])

df["flow_desc"] = (
    df["proto"].astype(str)
    + "|"
    + df["service"].astype(str)
    + "|"
    + df["state"].astype(str)
)

display(df[["proto", "service", "state", "flow_desc"]].head())


In [ ]:

pattern = (
    r"^(?P<p_proto>[^|]+)\|"
    r"(?P<p_service>[^|]+)\|"
    r"(?P<p_state>[^|]+)$"
)

parsed = df["flow_desc"].str.extract(pattern)

print(
    "Совпадение proto:",
    (parsed["p_proto"] == df["proto"]).mean()
)

print(
    "Совпадение service:",
    (parsed["p_service"] == df["service"]).mean()
)

print(
    "Совпадение state:",
    (parsed["p_state"] == df["state"]).mean()
)

display(parsed.head())


In [ ]:

df = df.drop(columns=["flow_desc"])

def clean_category(value):
    value = str(value).strip().lower()
    return re.sub(r"\s+", "_", value)

df["attack_cat"] = df["attack_cat"].apply(clean_category)

df["is_tcp"] = (
    df["proto"]
    .str.contains(r"^tcp$", regex=True, na=False)
    .astype(int)
)

top_proto = df["proto"].value_counts().head(5).index.tolist()

print("Топ-5 протоколов:")
display(df["proto"].value_counts().head(5).to_frame("count"))

df["proto_grp"] = df["proto"].where(
    df["proto"].isin(top_proto),
    "other"
)

print("Нормализованные категории атак:")
print(df["attack_cat"].unique())



# 4. Кодирование категориальных признаков

В работе используются **три метода**.

### 4.1 Label Encoding

Каждой категории назначается целое число.

### 4.2 Frequency Encoding

Категория заменяется долей её встречаемости.

### 4.3 One-Hot Encoding

Каждая категория превращается в отдельный бинарный столбец.

Для `proto` сначала используется укрупнённый признак `proto_grp`: пять наиболее частых протоколов сохраняются, остальные объединяются в `other`.


In [ ]:

le_state = LabelEncoder()
df["state_le"] = le_state.fit_transform(df["state"])

state_mapping = dict(
    zip(
        le_state.classes_,
        le_state.transform(le_state.classes_)
    )
)

print("Label Encoding state:")
print(state_mapping)

le_cat = LabelEncoder()
df["attack_cat_le"] = le_cat.fit_transform(df["attack_cat"])

attack_mapping = dict(
    zip(
        le_cat.classes_,
        le_cat.transform(le_cat.classes_)
    )
)

print("\nLabel Encoding attack_cat:")
print(attack_mapping)


In [ ]:

state_freq = df["state"].value_counts(normalize=True)
df["state_freq"] = df["state"].map(state_freq)

print("Frequency Encoding state:")
display(state_freq.round(4).to_frame("frequency"))


In [ ]:

shape_before_ohe = df.shape

df = pd.get_dummies(
    df,
    columns=["proto_grp", "service"],
    prefix=["proto", "svc"],
    dtype=int
)

print("До One-Hot:", shape_before_ohe)
print("После One-Hot:", df.shape)

display(df.head())



# 5. Визуализация данных

Ниже используется несколько различных способов визуального анализа:

1. тепловая карта пропусков;
2. `countplot` категорий атак;
3. гистограммы числовых признаков;
4. матрица корреляций;
5. `boxplot`;
6. stacked bar;
7. тепловая карта `service × attack`;
8. scatter plot;
9. отдельная гистограмма `sttl`.

Это позволяет исследовать как числовые, так и категориальные признаки.


In [ ]:

order = df["attack_cat"].value_counts().index

plt.figure(figsize=(11, 5))
sns.countplot(
    data=df,
    x="attack_cat",
    order=order
)
plt.xticks(rotation=45, ha="right")
plt.title("Рис. 2. Распределение категорий атак")
plt.xlabel("Категория атаки")
plt.ylabel("Количество")
plt.show()


In [ ]:

num_show = [
    "dur", "sbytes", "dbytes",
    "spkts", "dpkts", "rate",
    "sload", "dload", "smean"
]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))

for ax, col in zip(axes.flat, num_show):
    ax.hist(
        np.log1p(df[col].clip(lower=0)),
        bins=50
    )
    ax.set_title(f"log1p({col})")

fig.suptitle(
    "Рис. 3. Гистограммы числовых признаков",
    fontsize=14
)

plt.tight_layout()
plt.show()


In [ ]:

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if "attack_cat_le" in num_cols:
    num_cols.remove("attack_cat_le")

corr = df[num_cols].corr()

plt.figure(figsize=(18, 15))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0
)
plt.title("Рис. 4. Матрица корреляций Пирсона")
plt.show()


In [ ]:

high_corr = (
    corr.abs()
    .where(
        np.triu(
            np.ones(corr.shape),
            k=1
        ).astype(bool)
    )
    .stack()
    .sort_values(ascending=False)
)

print("Пары признаков с |r| > 0.95:")
display(high_corr[high_corr > 0.95].to_frame("|r|").head(30))

print("Признаки с наибольшей |корреляцией| с label:")
display(
    corr["label"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .to_frame("|r with label|")
)


In [ ]:

plt.figure(figsize=(12, 5))

sns.boxplot(
    data=df,
    x="attack_cat",
    y=np.log1p(df["sbytes"].clip(lower=0)),
    order=order
)

plt.xticks(rotation=45, ha="right")
plt.ylabel("log1p(sbytes)")
plt.xlabel("Категория атаки")
plt.title("Рис. 5. Размер переданных данных по типам атак")
plt.show()


In [ ]:

proto_cols = [
    col for col in df.columns
    if col.startswith("proto_")
]

proto_share = pd.DataFrame({
    col.replace("proto_", ""):
        df.loc[df[col] == 1, "label"]
        .value_counts(normalize=True)
    for col in proto_cols
}).T.fillna(0)

proto_share = proto_share.reindex(
    columns=[0, 1],
    fill_value=0
)

proto_share.columns = ["normal", "attack"]

display(proto_share.round(3))

proto_share.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5)
)

plt.ylabel("Доля")
plt.title("Рис. 6. Доля нормы и атак по группам протоколов")
plt.show()


In [ ]:

svc_cols = [
    col for col in df.columns
    if col.startswith("svc_")
]

svc_attack = pd.DataFrame({
    col.replace("svc_", ""):
        df.loc[df[col] == 1, "attack_cat"]
        .value_counts()
    for col in svc_cols
}).fillna(0)

plt.figure(figsize=(13, 7))

sns.heatmap(
    np.log1p(svc_attack),
    cmap="YlOrRd",
    annot=svc_attack.astype(int),
    fmt="d"
)

plt.title("Рис. 7. Сервис × тип атаки")
plt.xlabel("Сервис")
plt.ylabel("Категория атаки")
plt.show()


In [ ]:

sample_n = min(20_000, len(df))

sample = df.sample(
    sample_n,
    random_state=RANDOM_STATE
)

plt.figure(figsize=(8, 6))

plt.scatter(
    np.log1p(sample["sbytes"].clip(lower=0)),
    np.log1p(sample["dbytes"].clip(lower=0)),
    c=sample["label"],
    cmap="coolwarm",
    s=4,
    alpha=0.5
)

plt.xlabel("log1p(sbytes)")
plt.ylabel("log1p(dbytes)")
plt.title("Рис. 8. Диаграмма рассеяния sbytes–dbytes")
plt.show()


In [ ]:

plt.figure(figsize=(8, 5))

for label_value, name in [
    (0, "норма"),
    (1, "атака")
]:
    plt.hist(
        df.loc[
            df["label"] == label_value,
            "sttl"
        ],
        bins=60,
        alpha=0.6,
        label=name
    )

plt.legend()
plt.xlabel("sttl")
plt.ylabel("Количество")
plt.title("Рис. 9. Распределение sttl по классам")
plt.show()

print("sttl ∈ {254, 255}:")
display(
    df.loc[
        df["sttl"].isin([254, 255]),
        "label"
    ]
    .value_counts()
    .to_frame("count")
)



# 6. Создание и отбор признаков

Создаются четыре новых признака:

- `bytes_ratio` — отношение отправленных байт к принятым;
- `pkts_total` — общее количество пакетов;
- `bytes_per_pkt` — среднее количество байт на пакет;
- `ttl_diff` — абсолютная разница TTL.

Далее применяются:

1. логарифмирование сильно асимметричных признаков;
2. `VarianceThreshold`;
3. удаление сильно коррелирующих признаков;
4. `SelectKBest`;
5. `RFE`.

> **Важно:** в строках с логарифмированием сначала выполняется явное преобразование в `float64`.  
> Это предотвращает ошибку `LossySetitemError` в современных версиях Pandas.


In [ ]:

df["bytes_ratio"] = (
    df["sbytes"]
    / (df["dbytes"] + 1)
)

df["pkts_total"] = (
    df["spkts"]
    + df["dpkts"]
)

df["bytes_per_pkt"] = (
    (df["sbytes"] + df["dbytes"])
    / (df["pkts_total"] + 1)
)

df["ttl_diff"] = (
    df["sttl"] - df["dttl"]
).abs()

display(
    df[
        [
            "sbytes", "dbytes",
            "spkts", "dpkts",
            "sttl", "dttl",
            "bytes_ratio",
            "pkts_total",
            "bytes_per_pkt",
            "ttl_diff"
        ]
    ].head()
)


In [ ]:

drop_cols = [
    "proto",
    "state",
    "attack_cat",
    "attack_cat_le",
    "label"
]

X_all = df.drop(columns=drop_cols).copy()
y = df["label"].astype(int).copy()

non_numeric = (
    X_all
    .select_dtypes(exclude=[np.number])
    .columns
    .tolist()
)

if non_numeric:
    raise ValueError(
        "После кодирования остались строковые столбцы: "
        + ", ".join(non_numeric)
    )

print("Размер X_all:", X_all.shape)
print("Признаков до отбора:", X_all.shape[1])


In [ ]:

skew = X_all.skew(numeric_only=True).abs()

log_cols = skew[skew > 3].index.tolist()

log_cols = [
    col
    for col in log_cols
    if X_all[col].min() >= 0
    and X_all[col].nunique() > 2
]

for col in log_cols:
    X_all[col] = np.log1p(
        X_all[col].astype("float64")
    )

print(
    f"Логарифмировано {len(log_cols)} признаков:"
)
print(log_cols)



### 6.1 Удаление признаков с низкой дисперсией

Перед `VarianceThreshold` используется `MinMaxScaler`, чтобы все признаки были приведены к общей шкале `0..1`.

Порог:

```python
threshold=0.01
```

означает, что почти постоянные признаки будут удалены.


In [ ]:

X_mm = pd.DataFrame(
    MinMaxScaler().fit_transform(X_all),
    columns=X_all.columns,
    index=X_all.index
)

vt = VarianceThreshold(
    threshold=0.01
).fit(X_mm)

low_var = (
    X_all.columns[
        ~vt.get_support()
    ]
    .tolist()
)

print(
    f"Удалено по низкой дисперсии "
    f"({len(low_var)}):"
)
print(low_var)

X_sel = X_all.drop(
    columns=low_var
)

print(
    "Осталось:",
    X_sel.shape[1],
    "признаков"
)



### 6.2 Удаление мультиколлинеарных признаков

Если абсолютная корреляция двух признаков превышает `0.95`, один из них исключается.


In [ ]:

corr_m = X_sel.corr().abs()

upper = corr_m.where(
    np.triu(
        np.ones(corr_m.shape),
        k=1
    ).astype(bool)
)

multicol = [
    col
    for col in upper.columns
    if (upper[col] > 0.95).any()
]

print(
    f"Удалено мультиколлинеарных "
    f"({len(multicol)}):"
)
print(multicol)

X_sel = X_sel.drop(
    columns=multicol
)

print(
    "После удаления коррелирующих:",
    X_sel.shape
)



### 6.3 SelectKBest

Используется критерий `f_classif` (ANOVA F-test).  
Отбираются 20 признаков с наиболее выраженной статистической связью с целевым классом.


In [ ]:

k = min(20, X_sel.shape[1])

skb = SelectKBest(
    score_func=f_classif,
    k=k
).fit(X_sel, y)

scores = (
    pd.Series(
        skb.scores_,
        index=X_sel.columns
    )
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
    .sort_values(ascending=False)
)

kbest_feats = (
    scores.head(k)
    .index
    .tolist()
)

display(
    scores.head(20)
    .to_frame("F-score")
)


In [ ]:

plt.figure(figsize=(9, 7))

scores.head(k).sort_values().plot(
    kind="barh"
)

plt.title(
    "Рис. 10. SelectKBest: "
    f"{k} лучших признаков"
)

plt.xlabel("F-score")
plt.show()



### 6.4 Recursive Feature Elimination — RFE

Для RFE используется `DecisionTreeClassifier(max_depth=8)`.

Чтобы сократить время вычислений, RFE выполняется на случайной подвыборке до **30 000 строк**. Итоговые признаки формируются как объединение признаков, выбранных `SelectKBest` и `RFE`.


In [ ]:

sample_size = min(
    30_000,
    len(X_sel)
)

rng = np.random.default_rng(
    RANDOM_STATE
)

idx = rng.choice(
    len(X_sel),
    sample_size,
    replace=False
)

n_rfe = min(
    15,
    X_sel.shape[1]
)

rfe = RFE(
    estimator=DecisionTreeClassifier(
        max_depth=8,
        random_state=RANDOM_STATE
    ),
    n_features_to_select=n_rfe,
    step=3
)

rfe.fit(
    X_sel.iloc[idx],
    y.iloc[idx]
)

rfe_feats = (
    X_sel.columns[
        rfe.support_
    ]
    .tolist()
)

print(
    f"RFE выбрал {len(rfe_feats)} признаков:"
)
print(rfe_feats)


In [ ]:

final_feats = sorted(
    set(kbest_feats)
    | set(rfe_feats)
)

print(
    f"Итоговых признаков: "
    f"{len(final_feats)}"
)

print(final_feats)

X = X_sel[
    final_feats
].copy()

print("Итоговая матрица X:", X.shape)



# 7. Проверка сбалансированности классов

Для бинарной задачи анализируется `label`, а для понимания многоклассовой структуры — `attack_cat`.

Сильный дисбаланс может сделать обычную `accuracy` недостаточно информативной, поэтому в последующих лабораторных полезно использовать `precision`, `recall` и `F1-score`.


In [ ]:

label_counts = (
    y.value_counts()
    .sort_index()
)

label_share = (
    y.value_counts(
        normalize=True
    )
    .sort_index()
)

print("Количество объектов каждого label:")
display(
    label_counts.to_frame("count")
)

print("Доли:")
display(
    label_share
    .round(4)
    .to_frame("share")
)

print("attack_cat:")
display(
    df["attack_cat"]
    .value_counts(normalize=True)
    .round(4)
    .to_frame("share")
)


In [ ]:

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

label_counts.plot(
    kind="pie",
    autopct="%.1f%%",
    labels=[
        f"class {i}"
        for i in label_counts.index
    ],
    ax=axes[0]
)

axes[0].set_ylabel("")
axes[0].set_title("label")

df["attack_cat"].value_counts().plot(
    kind="bar",
    ax=axes[1],
    logy=True
)

axes[1].set_title(
    "attack_cat (log-шкала)"
)

fig.suptitle(
    "Рис. 11. Баланс классов"
)

plt.tight_layout()
plt.show()



# 8. Разбиение на X/y и train/test

`X` содержит итоговые признаки, `y` — целевую переменную `label`.

Используется стратифицированное разбиение:

```python
test_size=0.3
random_state=42
stratify=y
```

Поэтому доля атак должна быть практически одинаковой в `train` и `test`.


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y
)

print(
    "X_train:",
    X_train.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "y_test:",
    y_test.shape
)

print(
    "Доля атак train:",
    round(y_train.mean(), 4)
)

print(
    "Доля атак test:",
    round(y_test.mean(), 4)
)



# 9. Нормализация

Используется `StandardScaler`.

Критически важно:

```text
TRAIN → fit + transform
TEST  → только transform
```

То есть параметры масштабирования вычисляются **только на обучающей выборке**.  
Это предотвращает утечку информации из `test`.


In [ ]:

print("Диапазоны признаков до масштабирования:")

display(
    X_train
    .agg(["min", "max"])
    .T
)


In [ ]:

scaler = StandardScaler()

X_train_s = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=final_feats,
    index=X_train.index
)

X_test_s = pd.DataFrame(
    scaler.transform(X_test),
    columns=final_feats,
    index=X_test.index
)

print(
    "Максимальное |mean| train:",
    round(
        X_train_s
        .mean()
        .abs()
        .max(),
        6
    )
)

print(
    "std train:",
    round(
        X_train_s
        .std()
        .mean(),
        4
    )
)

print(
    "Диапазон mean test:",
    round(
        X_test_s
        .mean()
        .min(),
        4
    ),
    "...",
    round(
        X_test_s
        .mean()
        .max(),
        4
    )
)



## Сохранение подготовленных данных

Файлы создаются **во временном каталоге**, а не рядом с ноутбуком:

- `unsw_train_prepared.csv`
- `unsw_test_prepared.csv`

Они пригодны для следующей лабораторной работы, пока текущее ядро Jupyter работает.

Если их нужно сохранить постоянно, скопируйте их из `WORK_DIR` до выполнения последней ячейки очистки.


In [ ]:

TRAIN_PREPARED = (
    WORK_DIR
    / "unsw_train_prepared.csv"
)

TEST_PREPARED = (
    WORK_DIR
    / "unsw_test_prepared.csv"
)

X_train_s.assign(
    label=y_train
).to_csv(
    TRAIN_PREPARED,
    index=False
)

X_test_s.assign(
    label=y_test
).to_csv(
    TEST_PREPARED,
    index=False
)

print("Train сохранён:", TRAIN_PREPARED)
print("Test сохранён:", TEST_PREPARED)

print(
    "Размер train:",
    TRAIN_PREPARED.stat().st_size
    / 1024**2,
    "MB"
)

print(
    "Размер test:",
    TEST_PREPARED.stat().st_size
    / 1024**2,
    "MB"
)



# Итоговый вывод

В ходе лабораторной работы выполнен полный цикл предварительной обработки UNSW-NB15:

- набор данных загружен и исследован средствами Pandas;
- обнаружены скрытые пропуски в `service` и преобразованы в категорию `unknown`;
- выполнена работа со строками и регулярными выражениями;
- применены Label Encoding, Frequency Encoding и One-Hot Encoding;
- построены различные статистические визуализации;
- созданы новые сетевые признаки;
- применены `VarianceThreshold`, корреляционный отбор, `SelectKBest` и `RFE`;
- исследован баланс классов;
- сформированы `X` и `y`;
- выполнено стратифицированное разбиение `train/test`;
- признаки стандартизированы через `StandardScaler`.

Набор подготовлен к последующему обучению моделей машинного обучения.

### Методологическое замечание

Для сохранения структуры исходной лабораторной отбор признаков выполняется до `train_test_split`.

В промышленном ML-пайплайне ещё строже: процедуры, которые используют статистики данных или `y` (`SelectKBest`, RFE, масштабирование и т.п.), обычно обучают **только на train**, а к test применяют уже обученные преобразования. Это дополнительно защищает от `data leakage`.



# Очистка временных файлов

Эта ячейка удаляет:

- исходный `UNSW_NB15_training-set.csv`;
- подготовленные train/test CSV.

При обычной остановке ядра Jupyter очистка также зарегистрирована через `atexit`.

> Выполняйте эту ячейку только после того, как закончили работу с временными CSV.


In [ ]:

cleanup_temp_dir()
